# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [1]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

if not project_endpoint:
    raise ValueError("Please set AZURE_AI_PROJECT_ENDPOINT in your environment.")

client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model,
    credential=AzureCliCredential(),
)

c:\Work\agentsdemo\ai-agents-for-beginners\venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Work\agentsdemo\ai-agents-for-beginners\venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [2]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [4]:
agent = Agent(
    client=client,
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

Based on the list of destinations, here are some recommendations for warm beach vacation spots:

1. **Bali** - A tropical paradise offering beautiful beaches, luxury resorts, and vibrant culture.
2. **Cape Town** - Offers stunning beaches alongside incredible mountain views and a lively atmosphere.
3. **Rio de Janeiro** - A vibrant city known for its famous Copacabana and Ipanema beaches, as well as samba and Brazilian culture.
4. **Sydney** - Known for its iconic beaches like Bondi Beach and a mix of city and coastal experiences.

Would you like more details about any of these destinations?


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [5]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Tokyo, Japan's bustling capital, is a vibrant travel destination that uniquely blends traditional culture and cutting-edge modernity. Here's an overview of what makes Tokyo special:

### Highlights of Tokyo:
1. **Culture and Heritage:**
   - Explore ancient temples and shrines like the historic **Senso-ji Temple** in Asakusa and the tranquil **Meiji Shrine** in Shibuya.
   - Enjoy traditional performances like kabuki, Noh theater, or sumo wrestling matches.

2. **Modern Attractions:**
   - Discover futuristic architecture and technology in neighborhoods like **Shinjuku**, **Roppongi**, and the Akihabara district, which is a haven for anime, manga, and tech enthusiasts.
   - Visit iconic structures like the **Tokyo Skytree** or **Tokyo Tower** for panoramic city views.

3. **Culinary Experiences:**
   - Tokyo is a food lover's paradise, offering everything from world-class sushi in famous restaurants to casual comfort food such as ramen, yakitori, and street food.
   - Visit Tsukiji Mar

## Summary

In this lesson you learned how to:

- **Create a Foundry chat client** that connects to Azure AI Foundry via `FoundryChatClient`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.